# InstanSeg seed-threshold sweep on a small fixed crop

This notebook runs a controlled end-to-end sweep at `seed_threshold = [0.05, 0.1, 0.2, 0.4, 0.6]`. Every other inference setting matches M11 v4. It first writes one validated 4096-by-4096 OME-TIFF containing the previously reviewed removed-cell hotspot and the exact ten v4 segmentation channels in their configured order. All thresholds use that identical input, global normalization domain, tile geometry, watershed resolver, cleanup policy, and display field.

The earlier reconciliation notebook swept peak detection on precomputed seed maps; it did not rerun complete inference and is not equivalent to this experiment. Because this is a smaller WSI input, its global percentile bounds will be crop-derived rather than identical to the half-slide/full-slide bounds. They are nevertheless asserted identical across thresholds, so the sweep isolates threshold effects within this field. Run this notebook with the `instanseg_nimbus` kernel.

In [ ]:
from pathlib import Path
import inspect, json, re, shutil, subprocess, sys, time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
import zarr
from skimage.segmentation import find_boundaries

MIF_ROOT = Path('/data1/lowes/ratnayn/Codex/projects/mIF-pipeline')
INSTANSEG_ROOT = Path('/data1/lowes/ratnayn/Codex/projects/instanseg')
SOURCE = Path('/data1/lowes/ratnayn/Codex/codex-scratch/mIF-pipeline/instanseg_watershed_production_smoke_all_channel_crop/SLIDE-0330/SLIDE-0330_all_channels_half_crop.ome.tif')
OUTPUT_DIR = Path('/data1/lowes/ratnayn/Codex/codex-scratch/mIF-pipeline/instanseg_seed_threshold_sweep_small_crop/SLIDE-0330')
SWEEP_CROP = OUTPUT_DIR / 'SLIDE-0330_hotspot_4096_segmentation_channels.ome.tif'
SWEEP_SUMMARY_CSV = OUTPUT_DIR / 'SLIDE-0330_seed_threshold_sweep_summary.csv'
SWEEP_COMPARISON_PNG = OUTPUT_DIR / 'SLIDE-0330_seed_threshold_sweep_same_field.png'
SOURCE_CROP_BOUNDS = (22528, 26624, 11776, 15872)  # y0, y1, x0, x1; 4096 square
PRIOR_HOTSPOT_SOURCE_BOUNDS = (24748, 25172, 13818, 14243)
THRESHOLDS = (0.05, 0.1, 0.2, 0.4, 0.6)
RUN_CROP_WRITE = True
RUN_SWEEP = True
RUN_CLEANUP = True
REUSE_COMPLETE = True
OVERWRITE_CROP = False
OVERWRITE_INFERENCE = False
OVERWRITE_CLEANUP = False
MODEL_NAME = 'fluorescence_nuclei_and_cells'
PIXEL_SIZE_UM = 0.325
NORMALIZATION_PERCENTILES = (0.1, 99.9)
TILE_SIZE, OVERLAP, DETECTION_SIZE, BATCH_SIZE = 2048, 80, 20, 1
REFERENCE_CHANNEL = 'R1_DAPI'
SEGMENTATION_CHANNELS = [
    'R1_DAPI', 'R4_P19_POLYRAT', 'R4_GFP_POLY_AF488',
    'R6_CD45_CST_AF647', 'R6_PANCK_AE1_AE3_750',
    'R12_CD31_D8V9E_AF750', 'R7_NAK_ATPASE_555',
    'R8_F480_D2S9R_555', 'R9_CD68_E3O7V_488',
    'R12_CD3E_E4T1B_AF555',
]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert SOURCE_CROP_BOUNDS[0] <= PRIOR_HOTSPOT_SOURCE_BOUNDS[0] < PRIOR_HOTSPOT_SOURCE_BOUNDS[1] <= SOURCE_CROP_BOUNDS[1]
assert SOURCE_CROP_BOUNDS[2] <= PRIOR_HOTSPOT_SOURCE_BOUNDS[2] < PRIOR_HOTSPOT_SOURCE_BOUNDS[3] <= SOURCE_CROP_BOUNDS[3]
print({'source': str(SOURCE), 'crop': str(SWEEP_CROP), 'bounds': SOURCE_CROP_BOUNDS, 'thresholds': THRESHOLDS})

## 1. Write and validate one segmentation-channel sweep crop

In [ ]:
def ome_channel_names(xml):
    return [m.group(1) for m in re.finditer(r'<(?:[^:>]+:)?Channel\b[^>]*?Name="([^"]*)"', xml or '')]

def ome_float(xml, name):
    match = re.search(rf'{re.escape(name)}="([^"]+)"', xml or '')
    return float(match.group(1)) if match else None

def validate_crop():
    sy0, sy1, sx0, sx1 = SOURCE_CROP_BOUNDS
    with tifffile.TiffFile(SOURCE) as source_tif, tifffile.TiffFile(SWEEP_CROP) as crop_tif:
        source_series, crop_series = source_tif.series[0], crop_tif.series[0]
        expected = (len(SEGMENTATION_CHANNELS), sy1 - sy0, sx1 - sx0)
        assert source_series.axes == crop_series.axes == 'CYX'
        assert tuple(crop_series.shape) == expected and crop_series.dtype == np.uint16
        source_names = ome_channel_names(source_tif.ome_metadata)
        crop_names = ome_channel_names(crop_tif.ome_metadata)
        assert crop_names == SEGMENTATION_CHANNELS
        source_lookup = {name: index for index, name in enumerate(source_names)}
        source_channel_ids = [source_lookup[name] for name in SEGMENTATION_CHANNELS]
        source_pixel_size = (ome_float(source_tif.ome_metadata, 'PhysicalSizeY'), ome_float(source_tif.ome_metadata, 'PhysicalSizeX'))
        crop_pixel_size = (ome_float(crop_tif.ome_metadata, 'PhysicalSizeY'), ome_float(crop_tif.ome_metadata, 'PhysicalSizeX'))
        assert source_pixel_size == crop_pixel_size
        assert all(value is not None and abs(value - PIXEL_SIZE_UM) < 1e-3 for value in crop_pixel_size)
        source_store, crop_store = source_series.aszarr(level=0), crop_series.aszarr(level=0)
        try:
            source_array, crop_array = zarr.open(source_store, mode='r'), zarr.open(crop_store, mode='r')
            yy, xx = np.array([0, 1024, 2048, 3072, 4095]), np.array([0, 777, 2048, 3333, 4095])
            for crop_channel in (0, int(expected[0] // 2), expected[0] - 1):
                source_channel = source_channel_ids[crop_channel]
                observed = np.asarray(crop_array.oindex[crop_channel, yy, xx])
                reference = np.asarray(source_array.oindex[source_channel, sy0 + yy, sx0 + xx])
                assert np.array_equal(observed, reference)
        finally:
            source_store.close(); crop_store.close()
    return {'shape': expected, 'axes': 'CYX', 'dtype': 'uint16', 'channel_names': crop_names, 'source_channel_ids': source_channel_ids, 'pixel_size_yx_um': crop_pixel_size, 'sample_identity': True}

if SWEEP_CROP.exists() and OVERWRITE_CROP:
    print('[crop] removing the explicitly targeted prior sweep crop')
    SWEEP_CROP.unlink()
if SWEEP_CROP.exists():
    print('[crop] reusing existing crop after validation')
elif RUN_CROP_WRITE:
    sy0, sy1, sx0, sx1 = SOURCE_CROP_BOUNDS
    print('[crop] 1/3 reading 4096x4096 for the ten v4 segmentation channels')
    with tifffile.TiffFile(SOURCE) as tif:
        all_source_names = ome_channel_names(tif.ome_metadata)
        source_lookup = {name: index for index, name in enumerate(all_source_names)}
        missing = [name for name in SEGMENTATION_CHANNELS if name not in source_lookup]
        if missing: raise KeyError(f'Source is missing v4 segmentation channels: {missing}')
        selected_source_ids = [source_lookup[name] for name in SEGMENTATION_CHANNELS]
        source_pixel_y = ome_float(tif.ome_metadata, 'PhysicalSizeY')
        source_pixel_x = ome_float(tif.ome_metadata, 'PhysicalSizeX')
        if source_pixel_y is None or source_pixel_x is None:
            raise ValueError('Source OME-TIFF is missing physical pixel size metadata.')
        store = tif.series[0].aszarr(level=0)
        try:
            source_array = zarr.open(store, mode='r')
            crop_data = np.asarray(source_array.oindex[selected_source_ids, slice(sy0, sy1), slice(sx0, sx1)])
        finally:
            store.close()
    assert crop_data.dtype == np.uint16 and crop_data.shape[0] == len(SEGMENTATION_CHANNELS)
    print(f'[crop] 2/3 writing {crop_data.nbytes / 2**30:.2f} GiB tiled OME-TIFF')
    tifffile.imwrite(
        SWEEP_CROP, crop_data, ome=True, bigtiff=True, photometric='minisblack',
        tile=(512, 512), compression='zlib',
        metadata={'axes': 'CYX', 'Channel': {'Name': SEGMENTATION_CHANNELS},
                  'PhysicalSizeX': source_pixel_x, 'PhysicalSizeXUnit': 'µm',
                  'PhysicalSizeY': source_pixel_y, 'PhysicalSizeYUnit': 'µm'},
    )
    del crop_data
    print('[crop] 3/3 write complete; validating metadata and sampled source identity')
else:
    raise FileNotFoundError('Sweep crop does not exist and RUN_CROP_WRITE=False')
crop_validation = validate_crop()
print(crop_validation)

## 2. Resolve channels and verify the patched fork

In [ ]:
if str(INSTANSEG_ROOT) not in sys.path:
    sys.path.insert(0, str(INSTANSEG_ROOT))
import instanseg
from instanseg import InstanSeg
from tiffslide import TiffSlide
import instanseg.inference_class as inference_class
inference_class.TiffSlide = TiffSlide
fork_path = Path(instanseg.__file__).resolve()
fork_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=INSTANSEG_ROOT, text=True).strip()
assert fork_path.is_relative_to(INSTANSEG_ROOT)
assert hasattr(InstanSeg, 'eval_whole_slide_image_global_normalization')
with tifffile.TiffFile(SWEEP_CROP) as tif:
    crop_shape = tuple(int(v) for v in tif.series[0].shape)
    names = ome_channel_names(tif.ome_metadata)
lookup = {name: index for index, name in enumerate(names)}
missing = [name for name in SEGMENTATION_CHANNELS if name not in lookup]
if missing: raise KeyError(f'Missing segmentation aliases: {missing}')
CHANNEL_IDS = [lookup[name] for name in SEGMENTATION_CHANNELS]
REFERENCE_CHANNEL_ID = lookup[REFERENCE_CHANNEL]
assert CHANNEL_IDS[0] == REFERENCE_CHANNEL_ID
print({'python': sys.executable, 'instanseg': str(fork_path), 'commit': fork_commit, 'crop_shape': crop_shape, 'channel_ids': CHANNEL_IDS})

## 3. Run or safely reuse each WSI+watershed threshold

In [ ]:
def threshold_tag(value):
    return f'{float(value):.2f}'.replace('.', 'p')

def threshold_paths(value):
    root = OUTPUT_DIR / f'seed_{threshold_tag(value)}'
    return root, root / 'resolved.zarr', root / 'cleaned_8conn_min10.zarr'

def compatible_raw(path, threshold):
    if not path.is_dir(): return False
    try:
        attrs = dict(zarr.open(str(path), mode='r').attrs)
        settings, normalization = attrs.get('wsi_settings') or {}, attrs.get('normalization') or {}
        experiment = attrs.get('seed_sweep_experiment') or {}
        resolution = attrs.get('resolution') or {}
        return (attrs.get('status') == 'complete' and Path(attrs.get('source_image', '')).resolve() == SWEEP_CROP.resolve()
                and list(attrs.get('channel_ids', [])) == CHANNEL_IDS
                and settings.get('tile_size') == TILE_SIZE and settings.get('overlap') == OVERLAP
                and settings.get('detection_size') == DETECTION_SIZE
                and settings.get('resolve_cell_and_nucleus') is True and settings.get('resolution_method') == 'watershed'
                and [float(v) for v in normalization.get('percentiles', [])] == list(NORMALIZATION_PERCENTILES)
                and normalization.get('reference_channel_id') == REFERENCE_CHANNEL_ID
                and resolution.get('method') == 'watershed' and resolution.get('allow_unnucleated_cells') is True
                and float(experiment.get('seed_threshold', -1)) == float(threshold)
                and experiment.get('fork_commit') == fork_commit and experiment.get('fork_path') == str(fork_path)
                and experiment.get('model') == MODEL_NAME and experiment.get('channels') == SEGMENTATION_CHANNELS
                and float(experiment.get('pixel_size_um', -1)) == PIXEL_SIZE_UM
                and experiment.get('normalization_percentiles') == list(NORMALIZATION_PERCENTILES)
                and experiment.get('tile_size') == TILE_SIZE and experiment.get('overlap') == OVERLAP
                and experiment.get('detection_size') == DETECTION_SIZE and experiment.get('batch_size') == BATCH_SIZE
                and experiment.get('resolution_method') == 'watershed'
                and experiment.get('allow_unnucleated_cells') is True and experiment.get('cleanup_fragments') is True)
    except Exception:
        return False

raw_results, inference_minutes, model = {}, {}, None
for threshold in THRESHOLDS:
    run_dir, raw_path, _ = threshold_paths(threshold)
    run_dir.mkdir(parents=True, exist_ok=True)
    if compatible_raw(raw_path, threshold) and REUSE_COMPLETE:
        print(f'[seed={threshold}] reusing compatible completed inference')
    elif RUN_SWEEP:
        if raw_path.exists():
            if not OVERWRITE_INFERENCE:
                raise FileExistsError(f'Incompatible/partial output exists: {raw_path}')
            print(f'[seed={threshold}] removing explicitly targeted incompatible raw output')
            shutil.rmtree(raw_path)
            Path(str(raw_path) + '.normalization.json').unlink(missing_ok=True)
        print(f'[seed={threshold}] starting WSI inference')
        started = time.perf_counter()
        if model is None:
            model = InstanSeg(MODEL_NAME, verbosity=1)
        observed = model.eval_whole_slide_image_global_normalization(
            str(SWEEP_CROP), channel_ids=CHANNEL_IDS, pixel_size=PIXEL_SIZE_UM,
            normalization_percentiles=NORMALIZATION_PERCENTILES, reference_channel_id=REFERENCE_CHANNEL_ID,
            tile_size=TILE_SIZE, overlap=OVERLAP, detection_size=DETECTION_SIZE, batch_size=BATCH_SIZE,
            output_path=raw_path, overwrite=False, resolve_cell_and_nucleus=True,
            resolution_method='watershed', allow_unnucleated_cells=True,
            cleanup_fragments=True, seed_threshold=float(threshold),
        )
        assert Path(observed).resolve() == raw_path.resolve()
        inference_minutes[float(threshold)] = (time.perf_counter() - started) / 60
        stamped = zarr.open(str(raw_path), mode='r+')
        stamped.attrs['seed_sweep_experiment'] = {
            'seed_threshold': float(threshold), 'fork_commit': fork_commit, 'fork_path': str(fork_path), 'model': MODEL_NAME,
            'channels': SEGMENTATION_CHANNELS, 'pixel_size_um': PIXEL_SIZE_UM,
            'normalization_percentiles': list(NORMALIZATION_PERCENTILES), 'tile_size': TILE_SIZE,
            'overlap': OVERLAP, 'detection_size': DETECTION_SIZE, 'batch_size': BATCH_SIZE,
            'resolution_method': 'watershed', 'allow_unnucleated_cells': True, 'cleanup_fragments': True,
            'inference_minutes': inference_minutes[float(threshold)],
        }
    else:
        raise RuntimeError(f'No compatible output for seed={threshold}; set RUN_SWEEP=True')
    assert compatible_raw(raw_path, threshold)
    raw_results[float(threshold)] = zarr.open(str(raw_path), mode='r')
normalization_bounds = [raw_results[float(t)].attrs['normalization']['bounds'] for t in THRESHOLDS]
assert all(bounds == normalization_bounds[0] for bounds in normalization_bounds[1:])
print({'all_normalization_bounds_identical': True, 'bounds': normalization_bounds[0]})

## 4. Apply identical cleanup and assemble metrics

In [ ]:
helper_dir = MIF_ROOT / 'notebooks'
if str(helper_dir) not in sys.path: sys.path.insert(0, str(helper_dir))
import importlib
import instanseg_connectedness_cleanup as cleanup_module
cleanup_module = importlib.reload(cleanup_module)
run_cleanup = cleanup_module.run_cleanup
summaries, rows = {}, []
for threshold in THRESHOLDS:
    run_dir, raw_path, cleaned_path = threshold_paths(threshold)
    if not RUN_CLEANUP: raise RuntimeError('Set RUN_CLEANUP=True')
    print(f'[seed={threshold}] applying or reusing cleanup')
    cleanup_overwrite = OVERWRITE_CLEANUP
    if cleaned_path.exists() and zarr.open(str(cleaned_path), mode='r').attrs.get('status') != 'complete':
        cleanup_overwrite = True
        print(f'[seed={threshold}] rebuilding incomplete cleanup artifact')
    summary = run_cleanup(
        raw_path, cleaned_path, run_dir / 'cleanup_per_id.csv', run_dir / 'cleanup_summary.json',
        run_dir / 'removed_overview.png', SWEEP_CROP, reference_channel_id=REFERENCE_CHANNEL_ID,
        native_shape=crop_shape[-2:], chunk_size=1024, min_size=10,
        reuse=REUSE_COMPLETE, overwrite=cleanup_overwrite,
    )
    summaries[float(threshold)] = summary
    attrs = dict(raw_results[float(threshold)].attrs)
    validation = attrs.get('validation') or {}
    experiment = attrs.get('seed_sweep_experiment') or {}
    rows.append({
        'seed_threshold': float(threshold), 'inference_minutes': experiment.get('inference_minutes'),
        'raw_nuclei': validation.get('final_nuclei'), 'raw_cells': validation.get('final_cells'),
        'proxy_cells': validation.get('proxy_cells'),
        'unnucleated_cells': (validation.get('final_cells', 0) - validation.get('final_nuclei', 0)),
        'removed_nuclear_components': summary['removed_nuclear_components'],
        'removed_nuclear_pixels': summary['removed_nuclear_pixels'],
        'rejected_coordinated_ids': summary['rejected_coordinated_ids'],
        'removed_nucleus_free_cell_components': summary['removed_nucleus_free_cell_components'],
        'rejected_unnucleated_ids': summary['rejected_unnucleated_ids'],
        'removed_cell_pixels_total': summary['removed_cell_pixels_total'],
        'final_coordinated_ids': summary['final']['cells'],
        'final_nuclear_pixels': summary['final']['nuclear_foreground_pixels'],
        'final_cell_pixels': summary['final']['cell_foreground_pixels'],
        'cleanup_minutes': summary['elapsed_minutes'], 'all_cleanup_checks_pass': all(summary['checks'].values()),
    })
sweep_table = pd.DataFrame(rows).sort_values('seed_threshold')
sweep_table.to_csv(SWEEP_SUMMARY_CSV, index=False)
display(sweep_table)
assert sweep_table['all_cleanup_checks_pass'].all()

## 5. Same-field visual comparison for every threshold

In [ ]:
crop_y0, _, crop_x0, _ = SOURCE_CROP_BOUNDS
VIEW_BOUNDS = (PRIOR_HOTSPOT_SOURCE_BOUNDS[0] - crop_y0, PRIOR_HOTSPOT_SOURCE_BOUNDS[1] - crop_y0,
               PRIOR_HOTSPOT_SOURCE_BOUNDS[2] - crop_x0, PRIOR_HOTSPOT_SOURCE_BOUNDS[3] - crop_x0)
def label_view(array, bounds):
    y0, y1, x0, x1 = bounds; native_h, native_w = crop_shape[-2:]; model_h, model_w = array.shape[-2:]
    yy, xx = np.arange(y0, y1), np.arange(x0, x1)
    my = np.clip(((2 * yy + 1) * model_h) // (2 * native_h), 0, model_h - 1)
    mx = np.clip(((2 * xx + 1) * model_w) // (2 * native_w), 0, model_w - 1)
    sy0, sx0 = int(my.min()), int(mx.min())
    block = np.asarray(array[:, sy0:int(my.max()) + 1, sx0:int(mx.max()) + 1])
    return np.take(np.take(block, my - sy0, axis=1), mx - sx0, axis=2)
with tifffile.TiffFile(SWEEP_CROP) as tif:
    store = tif.series[0].aszarr(level=0)
    try:
        source = zarr.open(store, mode='r'); y0, y1, x0, x1 = VIEW_BOUNDS
        dapi = np.asarray(source.oindex[REFERENCE_CHANNEL_ID, slice(y0, y1), slice(x0, x1)], dtype=np.float32)
    finally: store.close()
low, high = np.percentile(dapi, (1, 99.8)); dapi = np.clip((dapi - low) / max(high - low, 1e-6), 0, 1)
fig, axes = plt.subplots(len(THRESHOLDS), 4, figsize=(18, 4.2 * len(THRESHOLDS)), squeeze=False)
for row, threshold in enumerate(THRESHOLDS):
    _, _, cleaned_path = threshold_paths(threshold)
    original = label_view(raw_results[float(threshold)], VIEW_BOUNDS)
    final = label_view(zarr.open(str(cleaned_path), mode='r'), VIEW_BOUNDS)
    removed_n = (original[0] > 0) & (final[0] == 0); removed_c = (original[1] > 0) & (final[1] == 0)
    cell_rgba = np.zeros(removed_c.shape + (4,), np.uint8); cell_rgba[removed_c] = (0, 80, 255, 120)
    nuc_rgba = np.zeros(removed_n.shape + (4,), np.uint8); nuc_rgba[removed_n] = (255, 0, 0, 230)
    for axis in axes[row]: axis.imshow(dapi, cmap='gray', interpolation='nearest'); axis.axis('off')
    axes[row, 0].set_title(f'seed={threshold} | DAPI')
    axes[row, 1].contour(find_boundaries(original[1], mode='outer'), [0.5], colors=['yellow'], linewidths=.45)
    axes[row, 1].contour(find_boundaries(original[0], mode='outer'), [0.5], colors=['cyan'], linewidths=.55)
    axes[row, 1].set_title('Original')
    axes[row, 2].imshow(cell_rgba); axes[row, 2].imshow(nuc_rgba)
    axes[row, 2].set_title(f'Removed: cell={int(removed_c.sum())} px; nucleus={int(removed_n.sum())} px')
    axes[row, 3].contour(find_boundaries(final[1], mode='outer'), [0.5], colors=['yellow'], linewidths=.45)
    axes[row, 3].contour(find_boundaries(final[0], mode='outer'), [0.5], colors=['cyan'], linewidths=.55)
    axes[row, 3].set_title('After cleanup')
fig.suptitle(f'Identical hotspot field for all thresholds | crop-native bounds {VIEW_BOUNDS}', fontsize=14)
fig.tight_layout(); fig.savefig(SWEEP_COMPARISON_PNG, dpi=180, bbox_inches='tight'); plt.show(); plt.close(fig)
print({'summary_csv': str(SWEEP_SUMMARY_CSV), 'comparison_png': str(SWEEP_COMPARISON_PNG), 'view_bounds': VIEW_BOUNDS})

In [ ]:
crop_y0, _, crop_x0, _ = [sum(x) for x in zip(SOURCE_CROP_BOUNDS, (0,750,500,0))]
VIEW_BOUNDS = (PRIOR_HOTSPOT_SOURCE_BOUNDS[0] - crop_y0, PRIOR_HOTSPOT_SOURCE_BOUNDS[1] - crop_y0,
               PRIOR_HOTSPOT_SOURCE_BOUNDS[2] - crop_x0, PRIOR_HOTSPOT_SOURCE_BOUNDS[3] - crop_x0)
def label_view(array, bounds):
    y0, y1, x0, x1 = bounds; native_h, native_w = crop_shape[-2:]; model_h, model_w = array.shape[-2:]
    yy, xx = np.arange(y0, y1), np.arange(x0, x1)
    my = np.clip(((2 * yy + 1) * model_h) // (2 * native_h), 0, model_h - 1)
    mx = np.clip(((2 * xx + 1) * model_w) // (2 * native_w), 0, model_w - 1)
    sy0, sx0 = int(my.min()), int(mx.min())
    block = np.asarray(array[:, sy0:int(my.max()) + 1, sx0:int(mx.max()) + 1])
    return np.take(np.take(block, my - sy0, axis=1), mx - sx0, axis=2)
with tifffile.TiffFile(SWEEP_CROP) as tif:
    store = tif.series[0].aszarr(level=0)
    try:
        source = zarr.open(store, mode='r'); y0, y1, x0, x1 = VIEW_BOUNDS
        dapi = np.asarray(source.oindex[REFERENCE_CHANNEL_ID, slice(y0, y1), slice(x0, x1)], dtype=np.float32)
    finally: store.close()
low, high = np.percentile(dapi, (1, 99.8)); dapi = np.clip((dapi - low) / max(high - low, 1e-6), 0, 1)
fig, axes = plt.subplots(len(THRESHOLDS), 4, figsize=(18, 4.2 * len(THRESHOLDS)), squeeze=False)
for row, threshold in enumerate(THRESHOLDS):
    _, _, cleaned_path = threshold_paths(threshold)
    original = label_view(raw_results[float(threshold)], VIEW_BOUNDS)
    final = label_view(zarr.open(str(cleaned_path), mode='r'), VIEW_BOUNDS)
    removed_n = (original[0] > 0) & (final[0] == 0); removed_c = (original[1] > 0) & (final[1] == 0)
    cell_rgba = np.zeros(removed_c.shape + (4,), np.uint8); cell_rgba[removed_c] = (0, 80, 255, 120)
    nuc_rgba = np.zeros(removed_n.shape + (4,), np.uint8); nuc_rgba[removed_n] = (255, 0, 0, 230)
    for axis in axes[row]: axis.imshow(dapi, cmap='gray', interpolation='nearest'); axis.axis('off')
    axes[row, 0].set_title(f'seed={threshold} | DAPI')
    axes[row, 1].contour(find_boundaries(original[1], mode='outer'), [0.5], colors=['yellow'], linewidths=.45)
    axes[row, 1].contour(find_boundaries(original[0], mode='outer'), [0.5], colors=['cyan'], linewidths=.55)
    axes[row, 1].set_title('Original')
    axes[row, 2].imshow(cell_rgba); axes[row, 2].imshow(nuc_rgba)
    axes[row, 2].set_title(f'Removed: cell={int(removed_c.sum())} px; nucleus={int(removed_n.sum())} px')
    axes[row, 3].contour(find_boundaries(final[1], mode='outer'), [0.5], colors=['yellow'], linewidths=.45)
    axes[row, 3].contour(find_boundaries(final[0], mode='outer'), [0.5], colors=['cyan'], linewidths=.55)
    axes[row, 3].set_title('After cleanup')
fig.suptitle(f'Identical hotspot field for all thresholds | crop-native bounds {VIEW_BOUNDS}', fontsize=14)
fig.tight_layout(); fig.savefig(SWEEP_COMPARISON_PNG, dpi=180, bbox_inches='tight'); plt.show(); plt.close(fig)
print({'summary_csv': str(SWEEP_SUMMARY_CSV), 'comparison_png': str(SWEEP_COMPARISON_PNG), 'view_bounds': VIEW_BOUNDS})

In [ ]:
# Large whole-crop QC: one full-width figure per seed threshold.
first_raw = raw_results[float(THRESHOLDS[0])]
model_h, model_w = first_raw.shape[-2:]
native_h, native_w = crop_shape[-2:]
model_y_in_native = np.clip(((2 * np.arange(model_h) + 1) * native_h) // (2 * model_h), 0, native_h - 1)
model_x_in_native = np.clip(((2 * np.arange(model_w) + 1) * native_w) // (2 * model_w), 0, native_w - 1)
with tifffile.TiffFile(SWEEP_CROP) as tif:
    store = tif.series[0].aszarr(level=0)
    try:
        source = zarr.open(store, mode='r')
        dapi_native = np.asarray(source[REFERENCE_CHANNEL_ID])
    finally:
        store.close()
dapi_whole = dapi_native[np.ix_(model_y_in_native, model_x_in_native)].astype(np.float32)
low, high = np.percentile(dapi_whole, (1, 99.8))
dapi_whole = np.clip((dapi_whole - low) / max(high - low, 1e-6), 0, 1)
del dapi_native

def show_outlines(axis, labels):
    cell_edge = find_boundaries(labels[1], mode='outer')
    nucleus_edge = find_boundaries(labels[0], mode='outer')
    overlay = np.zeros(cell_edge.shape + (4,), dtype=np.uint8)
    overlay[cell_edge] = (255, 220, 0, 220)
    overlay[nucleus_edge] = (0, 255, 255, 255)
    axis.imshow(overlay, interpolation='nearest')

for threshold in THRESHOLDS:
    run_dir, _, cleaned_path = threshold_paths(threshold)
    cleaned = zarr.open(str(cleaned_path), mode='r')
    if cleaned.attrs.get('status') != 'complete':
        raise RuntimeError(f'Cleanup is not complete for seed={threshold}: {cleaned_path}')
    original = np.asarray(raw_results[float(threshold)])
    final = np.asarray(cleaned)
    removed_n = (original[0] > 0) & (final[0] == 0)
    removed_c = (original[1] > 0) & (final[1] == 0)
    removed_rgba = np.zeros(removed_c.shape + (4,), dtype=np.uint8)
    removed_rgba[removed_c] = (0, 90, 255, 150)
    removed_rgba[removed_n] = (255, 0, 0, 245)
    fig, axes = plt.subplots(1, 4, figsize=(80, 20), dpi=130)
    for axis in axes:
        axis.imshow(dapi_whole, cmap='gray', interpolation='nearest')
        axis.axis('off')
    axes[0].set_title('Whole-crop DAPI', fontsize=18)
    show_outlines(axes[1], original)
    axes[1].set_title('Before cleanup: cells yellow, nuclei cyan', fontsize=18)
    show_outlines(axes[2], final)
    axes[2].set_title('After cleanup: cells yellow, nuclei cyan', fontsize=18)
    axes[3].imshow(removed_rgba, interpolation='nearest')
    axes[3].set_title(f'Removed: cell {int(removed_c.sum()):,} px; nucleus {int(removed_n.sum()):,} px', fontsize=18)
    fig.suptitle(f'Entire 4096 x 4096 crop | seed threshold {threshold}', fontsize=22)
    fig.tight_layout()
    whole_crop_png = run_dir / 'whole_crop_cleanup_qc.png'
    fig.savefig(whole_crop_png, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(whole_crop_png)
    del original, final, removed_n, removed_c, removed_rgba